# Notebook 02: Build Click-Level Tables (Train + Dev)

Goal: convert the raw impression format into a “flat” dataset:

**One row = one candidate article shown to a user**, with a label:
- `clicked = 1` if the user clicked
- `clicked = 0` otherwise

I save the processed tables to `data/processed/` so future notebooks can just **load** them instantly (no re-parsing).

### Imports

In [43]:
from pathlib import Path
import pandas as pd

###  Paths + output folder

In [44]:
PROJECT = Path().resolve().parent
DATA = PROJECT / "data"
PROCESSED = DATA / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

train_dir = DATA / "MINDsmall_train"
dev_dir   = DATA / "MINDsmall_dev"

print("PROJECT:", PROJECT)
print("PROCESSED:", PROCESSED)

PROJECT: C:\Users\jlsmp\Documents\universidade\M.IA\IAS\project
PROCESSED: C:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed


### Load raw TRAIN files

In [45]:
news_cols = ["news_id","category","subcategory","title","abstract","url","title_entities","abstract_entities"]
beh_cols  = ["impression_id","user_id","time","history","impressions"]

news_train = pd.read_table(train_dir / "news.tsv", header=None, names=news_cols)
beh_train  = pd.read_table(train_dir / "behaviors.tsv", header=None, names=beh_cols)

print("news_train:", news_train.shape)
print("beh_train:", beh_train.shape)

news_train: (51282, 8)
beh_train: (156965, 5)


### Build TRAIN click table (vectorized)

`behaviors.tsv` stores impressions as space-separated tokens like:

`N55689-1 N35729-0 ...`

So I:
1) split tokens into a list  
2) `explode()` into one row per token  
3) split `Nxxxx-1` into (`news_id`, `clicked`)

In [46]:
clicks = beh_train[["impression_id","user_id","time","history","impressions"]].copy()
clicks["impressions"] = clicks["impressions"].str.split()
clicks = clicks.explode("impressions", ignore_index=True)

tmp = clicks["impressions"].str.rsplit("-", n=1, expand=True)
clicks["news_id"] = tmp[0]
clicks["clicked"] = tmp[1].astype(int)

clicks = clicks.drop(columns=["impressions"])

print("clicks (train):", clicks.shape)
print("CTR (train):", clicks["clicked"].mean())
display(clicks.head(5))

clicks (train): (5843444, 6)
CTR (train): 0.040446010948338


,impression_id,user_id,time,history,news_id,clicked
0,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N55689,1
1,1,U13740,11/11/2019 9:05:58 AM,N55189 N42782 N34694 N45794 N18445 N63302 N104...,N35729,0
2,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N20678,0
3,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N39317,0
4,2,U91836,11/12/2019 6:11:30 PM,N31739 N6072 N63045 N23979 N35656 N43353 N8129...,N58114,0


### Join article metadata (category/subcategory/title)

Now each click row also has the article’s category information.

In [47]:
clicks = clicks.merge(
    news_train[["news_id","category","subcategory","title"]],
    on="news_id",
    how="left"
)

missing_meta = clicks["title"].isna().mean()
print("Missing metadata rate (train):", missing_meta)
display(clicks.sample(5, random_state=42))

Missing metadata rate (train): 0.0


,impression_id,user_id,time,history,news_id,clicked,category,subcategory,title
3874367,104139,U65183,11/14/2019 7:33:18 AM,N55714,N23805,0,lifestyle,awardstyle,Carrie Underwood rocks CMAs and several gorgeo...
4422490,118646,U66200,11/13/2019 3:00:42 PM,N138 N3909 N7333 N1150 N29177 N61070 N38342 N4...,N4642,0,music,music-celebrity,Kodak Black Sentenced to Over 3 Years in Priso...
147236,3886,U51286,11/13/2019 12:49:33 PM,N12098 N16363 N54889 N60340 N49048 N8834 N4579...,N50415,0,finance,markets,'Priceless' finds that turned out to be worthless
5023576,134854,U6813,11/10/2019 3:18:27 PM,N28296 N10235 N47671 N42620 N22270 N3091 N6224...,N26130,0,entertainment,humor,The highlights and lowlights from the world of...
5455306,146537,U52132,11/13/2019 11:02:51 AM,N33683 N44399 N38119 N44361 N6023 N46844 N2904...,N35233,0,lifestyle,lifestylebuzz,Groom Makes The Most Heartfelt Vows To His 9-Y...


### Validate TRAIN processing

These checks confirm I didn’t accidentally lose or duplicate impression tokens.

- `raw_tokens` = total number of `Nxxxx-0/1` tokens in the raw file
- `processed_rows` = number of rows after explode

In [48]:
raw_tokens = beh_train["impressions"].str.split().str.len().sum()
processed_rows = len(clicks)

print("raw_tokens:", raw_tokens)
print("processed_rows:", processed_rows)
assert raw_tokens == processed_rows, "Mismatch: parsing created wrong number of rows"

print("clicked value counts:")
print(clicks["clicked"].value_counts(dropna=False))

assert set(clicks["clicked"].unique()) <= {0, 1}, "clicked must be 0/1 only"
assert missing_meta < 1e-6, "Too many missing titles after merge (check split mismatch)" 

raw_tokens: 5843444
processed_rows: 5843444
clicked value counts:
clicked
0    5607100
1     236344
Name: count, dtype: int64


### Save processed TRAIN tables

I save:
- `clicks_train.pkl` (flat click table)
- `news_train.pkl` (raw article metadata)

Other notebooks will load these.

In [49]:
clicks.to_pickle(PROCESSED / "clicks_train.pkl")
news_train.to_pickle(PROCESSED / "news_train.pkl")
print("Saved train tables to:", PROCESSED)

Saved train tables to: C:\Users\jlsmp\Documents\universidade\M.IA\IAS\project\data\processed


### Repeat for DEV split

In [50]:
news_dev = pd.read_table(dev_dir / "news.tsv", header=None, names=news_cols)
beh_dev  = pd.read_table(dev_dir / "behaviors.tsv", header=None, names=beh_cols)

clicks_dev = beh_dev[["impression_id","user_id","time","history","impressions"]].copy()
clicks_dev["impressions"] = clicks_dev["impressions"].str.split()
clicks_dev = clicks_dev.explode("impressions", ignore_index=True)

tmp = clicks_dev["impressions"].str.rsplit("-", n=1, expand=True)
clicks_dev["news_id"] = tmp[0]
clicks_dev["clicked"] = tmp[1].astype(int)
clicks_dev = clicks_dev.drop(columns=["impressions"])

clicks_dev = clicks_dev.merge(
    news_dev[["news_id","category","subcategory","title"]],
    on="news_id",
    how="left"
)

print("clicks (dev):", clicks_dev.shape, "CTR (dev):", clicks_dev["clicked"].mean())
print("Missing dev metadata:", clicks_dev["title"].isna().mean())

clicks_dev.to_pickle(PROCESSED / "clicks_dev.pkl")
news_dev.to_pickle(PROCESSED / "news_dev.pkl")

print("Saved dev tables too.")

clicks (dev): (2740998, 9) CTR (dev): 0.040635928957263014
Missing dev metadata: 0.0
Saved dev tables too.
